# RF-DETR XAI - Environment Setup
Use the existing dermatoscopy_ai venv and verify required packages.

In [ ]:
from pathlib import Path
import os
import sys
import site

def find_repo_root(marker='pyproject.toml', max_depth=10):
    """Search upward from current directory for repo marker file."""
    current = Path.cwd().resolve()
    for _ in range(max_depth):
        if (current / marker).exists():
            return current
        current = current.parent
    return None

def prefer_venv_site_packages():
    """Keep venv site-packages before cluster/global package paths."""
    venv_prefix = str(Path(sys.prefix).resolve())
    venv_sites = [p for p in site.getsitepackages() if p.startswith(venv_prefix)]
    cleaned = [p for p in sys.path if not (p.startswith('/net/software/') and 'site-packages' in p)]
    for p in reversed(venv_sites):
        if p in cleaned:
            cleaned.remove(p)
        cleaned.insert(0, p)
    sys.path[:] = cleaned

    # If regex was imported earlier from cluster paths, evict it so next import uses venv.
    cached = sys.modules.get('regex')
    cached_path = getattr(cached, '__file__', '') if cached is not None else ''
    if cached_path.startswith('/net/software/'):
        for name in list(sys.modules.keys()):
            if name == 'regex' or name.startswith('regex.'):
                del sys.modules[name]

# Try to auto-locate repo; if it fails, set explicitly
REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    # Fallback: if auto-detect fails, use explicit path (EDIT if different)
    REPO_ROOT = Path('/net/tscratch/people/plgbmruszaj/dermatoscopy_ai')

RFD_XAI_ROOT = REPO_ROOT / 'dermato_ai' / 'rfdetr-xai'

if not RFD_XAI_ROOT.exists():
    print(f"ERROR: rfdetr-xai not found at {RFD_XAI_ROOT}")
    print("To fix: edit REPO_ROOT in this cell to the actual repo root path")
    raise FileNotFoundError(f"rfdetr-xai directory not found: {RFD_XAI_ROOT}")

os.chdir(RFD_XAI_ROOT)
sys.path.insert(0, str((RFD_XAI_ROOT / 'src').resolve()))
prefer_venv_site_packages()
print('✓ Repo root:', REPO_ROOT)
print('✓ rfdetr-xai root:', RFD_XAI_ROOT)
print('✓ Python executable:', sys.executable)

✓ Repo root: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai
✓ rfdetr-xai root: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/dermato_ai/rfdetr-xai
✓ Python executable: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/.venv/bin/python


In [6]:
!ls


experiments.md	notebooks  src	tests


In [7]:
import importlib

required = [
    'torch',
    'torchvision',
    'cv2',
    'pycocotools',
    'pandas',
    'matplotlib',
    'supervision',
    'tqdm',
]

for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        print(f'{pkg}: OK ({getattr(mod, '__version__', 'no-version')})')
    except Exception as exc:
        print(f'{pkg}: MISSING -> {exc}')

torch: OK (2.11.0+cu128)
torchvision: OK (0.26.0+cu128)
cv2: OK (4.10.0)
pycocotools: OK (no-version)
pandas: OK (2.3.3)
matplotlib: OK (3.10.8)
supervision: OK (0.27.0.post2)
tqdm: OK (4.67.3)


## Next
Run notebooks 02-06 in order. Edit dataset/checkpoint paths in each notebook before execution.